# DeepRacer-Genesis on Colab

AWS DeepRacer RL environment ported to [Genesis](https://github.com/Genesis-Embodied-AI/Genesis) — ROS-free, GPU-batched, vision-based, trained with rsl-rl PPO.

This notebook installs the package, trains a driving policy with PPO, validates the camera pipeline, and renders a bird's-eye video of many agents driving at once.

**Runtime**: needs a GPU runtime (T4 works; anything newer is faster). Each phase runs as a subprocess because Genesis initializes once per process.

In [ ]:
# @title GPU check
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

In [ ]:
# @title Install deepracer-genesis
# Change this to your fork if needed. For local development you can instead
# upload a wheel and `pip install /content/deepracer_genesis-*.whl`.
REPO = "https://github.com/USERNAME/deepracer-genesis.git"  # <-- EDIT ME
BRANCH = "main"

import glob
local_wheel = sorted(glob.glob("/content/deepracer_genesis-*.whl"))
if local_wheel:
    print("installing local wheel:", local_wheel[-1])
    !pip install -q {local_wheel[-1]}
else:
    !pip install -q "deepracer-genesis @ git+{REPO}@{BRANCH}"
!pip show deepracer-genesis | head -2

In [ ]:
# @title Fix gs-madrona vs newer CUDA toolkits
# gs-madrona 0.0.7 bundles nvJitLink 12.4 but compiles its megakernel with
# whatever NVRTC it finds (12.8 on Colab, 13.x elsewhere) -> LTO version
# mismatch -> `nvJitLink error: Internal error` at scene build. Giving it a
# matching NVRTC 12.4 on its $ORIGIN runpath fixes it.
import glob, os, pathlib, subprocess

subprocess.run(["pip", "install", "-q", "nvidia-cuda-nvrtc-cu12==12.4.127"], check=True)
sp = pathlib.Path(glob.glob("/usr/local/lib/python3*/dist-packages")[0])
if not (sp / "gs_madrona").exists():
    sp = pathlib.Path(glob.glob(os.path.expanduser("~/.local/lib/python3*/site-packages"))[0])
nvrtc = sp / "nvidia" / "cuda_nvrtc" / "lib"
mad = sp / "gs_madrona"
for name, target in [("libnvrtc.so.12", "libnvrtc.so.12"), ("libnvrtc.so", "libnvrtc.so.12"),
                     ("libnvrtc-builtins.so.12.4", "libnvrtc-builtins.so.12.4")]:
    link = mad / name
    link.unlink(missing_ok=True)
    link.symlink_to(nvrtc / target)
f = mad / "libmadgs_mgr.so"
b = f.read_bytes()
if b"libnvrtc.so.13" in b:
    f.write_bytes(b.replace(b"libnvrtc.so.13", b"libnvrtc.so.12"))
    print("patched dlopen name libnvrtc.so.13 -> .12")
print("madrona nvrtc fix applied")

In [ ]:
# @title Smoke test: physics env builds and steps
%%writefile /content/smoke.py
import torch
import genesis as gs
gs.init(backend=gs.cuda, logging_level="warning")
from deepracer_genesis.configs.cfgs import get_env_cfg
from deepracer_genesis.envs import DeepRacerEnv
env = DeepRacerEnv(num_envs=64, env_cfg=get_env_cfg(vision=False))
a = torch.zeros(64, 2, device=env.device)
import time
for _ in range(20):
    env.step(a)
t0 = time.perf_counter()
for _ in range(200):
    env.step(a)
print("physics OK:", round(64 * 200 / (time.perf_counter() - t0)), "steps/s aggregate")

In [ ]:
!python /content/smoke.py

In [ ]:
# @title Train a driving policy (state-based PPO, ~5-10 min on T4)
!cd /content && python -m deepracer_genesis.train -B 512 --max_iterations 50 --exp_name colab 2>&1 | grep -E "Mean reward|episode length|iteration" | tail -20

In [ ]:
# @title Camera validation (paired onboard + top-down images, 4 automated checks)
!cd /content && python -m deepracer_genesis.validation.camera_check --num_envs 4 --steps 60 2>&1 | grep -E "PASS|FAIL|==="

from IPython.display import Image as IPyImage, display
import glob
for p in sorted(glob.glob("/content/logs/validation/env0_*_onboard.png"))[:1] + \
         sorted(glob.glob("/content/logs/validation/env0_*_topdown.png"))[:1]:
    print(p)
    display(IPyImage(p, width=320))

In [ ]:
# @title Render the many-agents demo video (spectator, all cars at once)
import glob
ckpt = sorted(glob.glob("/content/logs/colab/model_*.pt"),
              key=lambda p: int(p.rsplit("_", 1)[1][:-3]))[-1]
print("checkpoint:", ckpt)
!cd /content && python -m deepracer_genesis.eval --checkpoint {ckpt} --num_envs 16 --steps 500 --res 960x720 --out logs/eval 2>&1 | tail -3

In [ ]:
# @title Watch it
from IPython.display import Video
Video("/content/logs/eval/spectator.mp4", embed=True, width=720)

## Going further

- **Vision policy** (CNN on the 160x120 onboard camera, Madrona batch renderer):
  `!python -m deepracer_genesis.train -B 128 --vision --max_iterations 300 --exp_name vision`
- **Domain randomization** (per-env friction/mass/COM/gains/armature, per the
  [Genesis DR guide](https://genesis-world.readthedocs.io/en/latest/user_guide/getting_started/policy_training/best_practices/domain_randomization.html)): add `--randomize`.
- **Heterogeneous tracks** (each env a different track):
  `!python -m deepracer_genesis.validation.camera_check --num_envs 6 --tracks reinvent_base,reInvent2019_track,2022_reinvent_champ`
- **Throughput table**: `!python benchmarks/throughput.py --sweep` (clone the repo for the benchmarks/ dir).
- Longer training: bump `--max_iterations` (300+ gives confident lapping).
- Branches: `raster-vision` (color-faithful rasterizer obs), `nyx-vision`
  (photorealistic Nyx renderer; needs driver 575+), `cpu-vision` (no GPU).